# 04 · Train your own small object detector (optional)

YOLO-World already finds the eight objects, but it is big. This notebook makes a
small, fast YOLOv8n that knows exactly these eight, **without hand-labelling**:

1. take pictures with the webcam, where the device will be used;
2. YOLO-World labels them automatically (review a few!);
3. YOLOv8n trains on the GTX 1650.

The same pictures, at 96×96, are what the wearable's FOMO model needs too.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
import cv2, time
from va_lab.vision import open_camera
def capture(n=60, every_s=1.0, folder=config.IMAGES_DIR / "raw"):
    folder.mkdir(parents=True, exist_ok=True); cap = open_camera(); saved = 0; last = 0
    while saved < n:
        ok, frame = cap.read(); cv2.imshow("capturing (Q stops)", frame)
        if time.time() - last >= every_s:
            cv2.imwrite(str(folder / f"img_{int(time.time() * 1000)}.jpg"), frame); saved += 1; last = time.time()
        if cv2.waitKey(1) & 0xFF in (ord("q"), 27): break
    cap.release(); cv2.destroyAllWindows(); print("saved", saved, "pictures in", folder)
capture(n=60)   # walk around: doors, chairs, tables, bottles, people (with consent)

In [ ]:
import random, shutil, yaml, pathlib
from va_lab.vision import Detector
det = Detector("world")
names = config.OBJECTS
root = config.IMAGES_DIR / "yolo"
for s in ("train", "val"):
    (root / "images" / s).mkdir(parents=True, exist_ok=True); (root / "labels" / s).mkdir(parents=True, exist_ok=True)
imgs = sorted((config.IMAGES_DIR / "raw").glob("*.jpg")); random.seed(0); random.shuffle(imgs)
n_boxes = 0
for i, p in enumerate(imgs):
    split = "val" if i % 5 == 0 else "train"
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    lines = []
    for d in det.detect(img, min_score=0.45):
        x1, y1, x2, y2 = d.box
        lines.append(f"{names.index(d.obj)} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
    shutil.copy(p, root / "images" / split / p.name)
    (root / "labels" / split / (p.stem + ".txt")).write_text("\n".join(lines))
    n_boxes += len(lines)
(root / "data.yaml").write_text(yaml.safe_dump({"path": str(root), "train": "images/train", "val": "images/val",
                                                "names": {i: n for i, n in enumerate(names)}}))
print(len(imgs), "pictures,", n_boxes, "auto-labelled boxes ->", root / "data.yaml")

In [ ]:
from ultralytics import YOLO
y = YOLO(str(config.MODELS / "yolov8n.pt"))
res = y.train(data=str(config.IMAGES_DIR / "yolo" / "data.yaml"), epochs=50, imgsz=320, batch=16,
              device=0, project=str(config.MODELS / "runs"), name="objects", exist_ok=True)
best = pathlib.Path(res.save_dir) / "weights" / "best.pt"
shutil.copy(best, config.MODELS / "objects_custom.pt"); print("saved", config.MODELS / "objects_custom.pt")

In [ ]:
# Use it: python -m va_lab.assistant --detector custom
YOLO(str(config.MODELS / "objects_custom.pt")).export(format="onnx", imgsz=320)